# Vorbereitung des Teilexperiment B

Aufbereitung der Daten für den zweiten Experimentendurchlauf. Verwendung des Dataset vrdu ad-buy-form gemischt mit dem DocILE Korpus zur externen Validierung der Experimentbedingungen auf Daten, die einen echten Einsatz simulieren.

In [2]:
import os

RAW_DATA_PATH = "../data/raw/vrdu/ad-buy-form/"
RAW_DATA_PATH_DOCILE = "../data/raw/docile"
OUTPUT_DIR = "../data/processed/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
import pandas as pd

df_ad_buy_form = pd.read_json(
    "../data/raw/vrdu/ad-buy-form/main/dataset.jsonl", lines=True
)

df_ad_buy_form.head(5)

,filename,file_path,ocr,annotations
0,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,./pdfs/00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,{'text': 'Page 1 of 2 INVOICE FOX Remit Addres...,"[[property, [['KMSP\n', [0, 0.14883721, 0.0662..."
1,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,./pdfs/00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,{'text': 'Page 1 of 1 INVOICE DUE ATE 2436316-...,"[[property, [['WXXA\n', [0, 0.14763948, 0.0677..."
2,00c3353e-a25f-574a-a9db-39a41579895a.pdf,./pdfs/00c3353e-a25f-574a-a9db-39a41579895a.pdf,{'text': 'Print Date 02/28/20 14:21:20 Page 1 ...,"[[contract_num, [['14086\n', [0, 0.27917981000..."
3,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,./pdfs/01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,{'text': 'Page 1 of 4 INVOICE Advertiser 4 Rem...,"[[property, [['KCNC-TV\n', [0, 0.14883721, 0.0..."
4,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,./pdfs/0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,{'text': 'Contract # Date Entered Schedule Dat...,"[[contract_num, [['575746\n', [0, 0.3472609200..."


In [4]:
import json
import pandas as pd
from pathlib import Path


def create_dataframe_from_json(json_path, pdf_dir, ocr_dir, ann_dir):
    with open(RAW_DATA_PATH_DOCILE + json_path, "r") as f:
        ids = json.load(f)

    data_list = []

    for file_id in ids:
        pdf_file = Path(RAW_DATA_PATH_DOCILE + pdf_dir) / f"{file_id}.pdf"
        ocr_file = Path(RAW_DATA_PATH_DOCILE + ocr_dir) / f"{file_id}.json"
        ann_file = Path(RAW_DATA_PATH_DOCILE + ann_dir) / f"{file_id}.json"

        if ocr_file.exists() and ann_file.exists():
            with open(ocr_file, "r") as f_ocr:
                ocr_content = json.load(f_ocr)
            with open(ann_file, "r") as f_ann:
                ann_content = json.load(f_ann)

            data_list.append(
                {
                    "doc_id": file_id,
                    "file_path": str(pdf_file),
                    "ocr": ocr_content,
                    "annotations": ann_content,
                }
            )

    return pd.DataFrame(data_list)


df_docile = create_dataframe_from_json("/train.json", "/pdfs", "/ocr", "/annotations")

In [5]:
df_docile.head()

,doc_id,file_path,ocr,annotations
0,00134dd365a24343b35b78c6,../data/raw/docile/pdfs/00134dd365a24343b35b78...,"{'pages': [{'page_idx': 0, 'dimensions': [1616...",{'field_extractions': [{'bbox': [0.78253089891...
1,00136a27c7774c1e8dc6b2f2,../data/raw/docile/pdfs/00136a27c7774c1e8dc6b2...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.63452136714...
2,002e3cf97973428f905671b3,../data/raw/docile/pdfs/002e3cf97973428f905671...,"{'pages': [{'page_idx': 0, 'dimensions': [1689...",{'field_extractions': [{'bbox': [0.72741935483...
3,002f9b82b74f4258b3b072d0,../data/raw/docile/pdfs/002f9b82b74f4258b3b072...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.49176869947...
4,003568b1286f4dab953fc2d5,../data/raw/docile/pdfs/003568b1286f4dab953fc2...,"{'pages': [{'page_idx': 0, 'dimensions': [1684...",{'field_extractions': [{'bbox': [0.82242516938...


### Preprocess both Datasets

VRDU:

In [12]:
import pandas as pd

# Copied from data_preperation_A_internal_validation.ipynb
def extract_clean_text(data):
    if isinstance(data, str):
        return data.replace("\n", " ").strip()
    elif isinstance(data, list):
        texts = []
        for item in data:
            # Bounding-Boxen überspringen (Listen, die mit Zahlen beginnen)
            if (
                isinstance(item, list)
                and len(item) > 0
                and isinstance(item[0], (int, float))
            ):
                continue
            extracted = extract_clean_text(item)
            if extracted:
                texts.append(extracted)
        return " ".join(texts)
    return ""


def create_vrdu_dataframe(df_input):
    processed_docs = []

    for doc in df_input.to_dict(orient="records"):
        annotations_list = doc.get("annotations", [])
        ground_truth = {}
        line_items_extracted = []

        if isinstance(annotations_list, list):
            for annotation in annotations_list:
                if not isinstance(annotation, list) or len(annotation) < 2:
                    continue

                entity_keys = annotation[0]
                entity_values = annotation[1]

                if isinstance(entity_keys, str):
                    field_name = entity_keys
                    clean_value = extract_clean_text(entity_values)

                    if field_name in ground_truth and ground_truth[field_name]:
                        ground_truth[field_name] += " " + clean_value
                    else:
                        ground_truth[field_name] = clean_value

                elif isinstance(entity_keys, list):
                    headers = entity_keys
                    rows = entity_values

                    if isinstance(rows, list):
                        for row in rows:
                            row_dict = {}
                            if isinstance(row, list):
                                for idx, header in enumerate(headers):
                                    if idx < len(row):
                                        row_dict[header] = extract_clean_text(row[idx])
                                line_items_extracted.append(row_dict)

        if line_items_extracted:
            ground_truth["line_items"] = line_items_extracted
            has_line_items = True
        else:
            has_line_items = False

        property_clean = ground_truth.get("property", "UNKNOWN")

        processed_docs.append(
            {
                "doc_id": doc.get("filename", "unknown_id"),
                "property": property_clean,
                "has_line_items": has_line_items,
                "ground_truth": ground_truth,
                "raw_doc": doc,
                "annotations": annotations_list,
                "ocr_text": extract_clean_text(doc.get("ocr", {}).get("text", "")),
            }
        )

    return pd.DataFrame(processed_docs)


df_ad_buy_form = create_vrdu_dataframe(df_ad_buy_form)

In [13]:
df_ad_buy_form.head()

,doc_id,property,has_line_items,ground_truth,raw_doc,annotations,ocr_text
0,00a83bbc-0101-f092-4bd7-e75f315e8f14.pdf,KMSP KMSP KMSP KMSP,True,"{'property': 'KMSP KMSP KMSP KMSP', 'tv_addres...",{'filename': '00a83bbc-0101-f092-4bd7-e75f315e...,"[[property, [['KMSP\n', [0, 0.14883721, 0.0662...",Page 1 of 2 INVOICE FOX Remit Address: KMSP 46...
1,00c29ad8-c88b-b3bb-1a39-3267e47c7a88.pdf,WXXA WXXA,True,"{'property': 'WXXA WXXA', 'tv_address': 'PO Bo...",{'filename': '00c29ad8-c88b-b3bb-1a39-3267e47c...,"[[property, [['WXXA\n', [0, 0.14763948, 0.0677...",Page 1 of 1 INVOICE DUE ATE 2436316-1 POL/FWD....
2,00c3353e-a25f-574a-a9db-39a41579895a.pdf,WSIL-TV,True,"{'contract_num': '14086', 'product': 'Mike Car...",{'filename': '00c3353e-a25f-574a-a9db-39a41579...,"[[contract_num, [['14086\n', [0, 0.27917981000...",Print Date 02/28/20 14:21:20 Page 1 of 1 ORDER...
3,01250d60-2a0a-0c93-5a18-9f1f195fdc1a.pdf,KCNC-TV KCNC-TV KCNC-TV KCNC-TV KCNC-TV KCNC-T...,True,{'property': 'KCNC-TV KCNC-TV KCNC-TV KCNC-TV ...,{'filename': '01250d60-2a0a-0c93-5a18-9f1f195f...,"[[property, [['KCNC-TV\n', [0, 0.14883721, 0.0...",Page 1 of 4 INVOICE Advertiser 4 Remit Address...
4,0179ed96-6b64-eb8b-07c1-3214262dec0c.pdf,KBJR Television Inc,True,"{'contract_num': '575746', 'flight_from': '05/...",{'filename': '0179ed96-6b64-eb8b-07c1-3214262d...,"[[contract_num, [['575746\n', [0, 0.3472609200...",Contract # Date Entered Schedule Dates Last Mo...


In [ ]:
# Kopiert von data_preperation_A_internal_validation.ipynb
def count_annotation_spans(annotations_list):
    """
    Zählt die Gesamtanzahl aller annotierten Werteinstanzen in einem Dokument.
    - Flat fields: zählt jeden OCR-Span
    - Line items:  zählt jede Tabellenzeile
    """
    if not isinstance(annotations_list, list):
        return 0
    count = 0
    for ann in annotations_list:
        if not isinstance(ann, list) or len(ann) < 2:
            continue
        if isinstance(ann[0], str):  # Flat field
            count += len(ann[1]) if isinstance(ann[1], list) else 1
        elif isinstance(ann[0], list):  # Line item (Hierarchical)
            count += len(ann[1]) if isinstance(ann[1], list) else 1
    return count


df_ad_buy_form["annotation_count"] = df_ad_buy_form["raw_doc"].apply(
    lambda doc: count_annotation_spans(doc.get("annotations", []))
)

# Terzile berechnen
p33 = df_ad_buy_form["annotation_count"].quantile(0.33)
p66 = df_ad_buy_form["annotation_count"].quantile(0.66)

df_ad_buy_form["complexity"] = df_ad_buy_form["annotation_count"].apply(
    lambda c: "L1" if c <= p33 else ("L2" if c <= p66 else "L3")
)

print(df_ad_buy_form["complexity"].value_counts())
print(f"\nTerzilgrenzen: L1 ≤ {p33:.0f}, L2 ≤ {p66:.0f}, L3 > {p66:.0f}")
print(df_ad_buy_form.groupby("complexity")["annotation_count"].describe())

complexity
L1    1878
L3    1731
L2    1571
Name: count, dtype: int64

Terzilgrenzen: L1 <= 19, L2 <= 38, L3 > 38


DOCILE:

In [7]:
def ocr_to_text_docile(daten):
    clean_text = ""
    for page in daten.get("pages", []):
        for block in page.get("blocks", []):
            for line in block.get("lines", []):
                worte = [word.get("value", "") for word in line.get("words", [])]
                clean_text += " ".join(worte) + "\n"
            clean_text += "\n"
    return clean_text

df_docile["ocr_text"] = df_docile["ocr"].apply(ocr_to_text_docile)

In [8]:
def extract_ground_truth_docile(annotations):
    ground_truth = {}
    if isinstance(annotations, dict):
        extractions = annotations.get("field_extractions", [])
        for field in extractions:
            fieldtype = field.get("fieldtype")
            text = field.get("text", "").strip()

            if fieldtype in ground_truth:
                ground_truth[fieldtype] += " " + text
            else:
                ground_truth[fieldtype] = text

    return ground_truth


df_docile["ground_truth"] = df_docile["annotations"].apply(extract_ground_truth_docile)

In [9]:
df_docile.head(5)

,doc_id,file_path,ocr,annotations,ocr_text,ground_truth
0,00134dd365a24343b35b78c6,../data/raw/docile/pdfs/00134dd365a24343b35b78...,"{'pages': [{'page_idx': 0, 'dimensions': [1616...",{'field_extractions': [{'bbox': [0.78253089891...,"WIRE INSTRUCTIONS:\nNationsBank, N.A.\nBaltimo...","{'document_id': '990304502', 'date_issue': '03..."
1,00136a27c7774c1e8dc6b2f2,../data/raw/docile/pdfs/00136a27c7774c1e8dc6b2...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.63452136714...,Please Remit To:\nKMOZ 92.3 The Moose\n1360 E....,"{'document_id': '17685-2 17685-2', 'date_issue..."
2,002e3cf97973428f905671b3,../data/raw/docile/pdfs/002e3cf97973428f905671...,"{'pages': [{'page_idx': 0, 'dimensions': [1689...",{'field_extractions': [{'bbox': [0.72741935483...,"UMI PUBLICATIONS, INC.\nP.O.BOX 30036 (28230)\...","{'document_id': '096084', 'date_issue': '12/09..."
3,002f9b82b74f4258b3b072d0,../data/raw/docile/pdfs/002f9b82b74f4258b3b072...,"{'pages': [{'page_idx': 0, 'dimensions': [1584...",{'field_extractions': [{'bbox': [0.49176869947...,Page 1 of1\n\nINVOICE\nInvoice #\n2081683-1\nS...,"{'document_id': '2081683-1', 'date_issue': '09..."
4,003568b1286f4dab953fc2d5,../data/raw/docile/pdfs/003568b1286f4dab953fc2...,"{'pages': [{'page_idx': 0, 'dimensions': [1684...",{'field_extractions': [{'bbox': [0.82242516938...,Appud 10032089\n\nWAIT\nKLEN\nSAINISIC\nWalt K...,"{'document_id': '23299 23299', 'date_issue': '..."


In [18]:
def count_docile_annotations(annotations_dict):
    if not isinstance(annotations_dict, dict):
        return 0

    count = len(annotations_dict.get("field_extractions", []))

    # Berücksichtigt auch Tabellenzeilen, falls diese separat in DOCILE existieren
    count += len(annotations_dict.get("line_item_extractions", []))

    return count


df_docile["annotation_count"] = df_docile["annotations"].apply(count_docile_annotations)

p33 = df_docile["annotation_count"].quantile(0.33)
p66 = df_docile["annotation_count"].quantile(0.66)

df_docile["complexity"] = df_docile["annotation_count"].apply(
    lambda c: "L1" if c <= p33 else ("L2" if c <= p66 else "L3")
)

print(df_docile["complexity"].value_counts())
print(f"\nTerzilgrenzen: L1 <= {p33:.0f}, L2 <= {p66:.0f}, L3 > {p66:.0f}")

complexity
L1    1878
L3    1731
L2    1571
Name: count, dtype: int64

Terzilgrenzen: L1 <= 19, L2 <= 38, L3 > 38


# Auswertung des Splits

### Ziehung der Stichprobe

## Statistische Auswertung der Stichprobenziehung

### Verteilung der Daten


### Statistischer Test der Unterschiede zwischen Annotationen in den Gruppen


### Prüfung der relativen Informationsdichte

### Prüfung der Anzahl von line_items pro Komplexität

### Erstellung des Korpus für Teilexperiment B

In [10]:
experiment_documents = {}

In [11]:
with open("../data/processed/corpus_experiment_B_external.jsonl", "w") as f:
    for entry in experiment_documents:
        f.write(entry.to_json(ensure_ascii=False) + "\n")